# 0 - Package

In [17]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from google.colab import files
from typing import Optional, Tuple


In [4]:
# Helper

def export_df_to_excel(
    df: pd.DataFrame,
    file_path: str,
    sheet_name: str = "data",
    index: bool = True,
):
    df.to_excel(
        file_path,
        sheet_name=sheet_name,
        index=index
    )


# 1 - Importer Indice historique


In [5]:
def load_excel_file():
    """
    Importe un fichier Excel depuis Colab et retourne :
    - le DataFrame complet
    - la colonne Date convertie en datetime
    - la colonne Index convertie en numérique
    """

    uploaded = files.upload()
    filename = next(iter(uploaded))

    df = pd.read_excel(filename)

    dates = pd.to_datetime(df["Date"])

    index = pd.to_numeric(
        df["Index"].astype(str).str.replace(",", "."),
        errors="coerce"
    )

    return df, dates, index

In [6]:
df1, dates1, index1 = load_excel_file()
# df2, dates2, index2 = load_excel_file()
# df3, dates3, index3 = load_excel_file()

print(df1.head())
#print(df2.head())

Saving Data.xlsx to Data.xlsx
        Date        Index
0 2025-12-31  1000.000000
1 2026-01-02   999.726357
2 2026-01-05  1006.618624
3 2026-01-06  1013.130545
4 2026-01-07  1014.727224


# 2 - Traitement de Time Serie 数据TS清洗


In [12]:
def load_index_series_df(df: pd.DataFrame) -> pd.Series:
    """
    精简版：加载并清洗包含日期和指数点位的 DataFrame，返回带 DatetimeIndex 的 Series。
    """
    date_col, level_col = df.columns[:2]

    # 使用链式调用一步完成转换、去重与索引设置
    s = (
        df.assign(
            date=pd.to_datetime(df[date_col], errors="coerce"),
            val=pd.to_numeric(
                df[level_col].astype(str).str.replace(" ", "", regex=False).str.replace(",", ".", regex=False),
                errors="coerce"
            )
        )
        .dropna(subset=['date', 'val'])
        .drop_duplicates(subset='date')
        .set_index('date')['val']
        .sort_index()
    )

    if s.empty:
        raise ValueError("Series vide après nettoyage.")
    if (s <= 0).any():
        raise ValueError("Certaines valeurs sont <= 0.")

    return s

In [13]:
levels = load_index_series_df(df1)
print(levels)
print("--- 检查 DataFrame 的数据类型 ---")
print(levels.dtypes)

date
2025-12-31    1000.000000
2026-01-02     999.726357
2026-01-05    1006.618624
2026-01-06    1013.130545
2026-01-07    1014.727224
                 ...     
2026-09-09    1129.565067
2026-09-10    1122.181018
2026-09-11    1132.994206
2026-09-14    1126.685366
2026-09-15    1117.872007
Name: val, Length: 177, dtype: float64
--- 检查 DataFrame 的数据类型 ---
float64


#3 - Black&Schole Model

In [15]:
@dataclass
class BSParams:
    r_annual_cc: float
    sigma_annual: float
    s0: float
    s0_date: Optional[pd.Timestamp]

##  3.1 BS Vol imp estimation

In [14]:
def estimate_sigma_from_history(levels: pd.Series, day_count: int = 365) -> float:
    """
    计算资产的历史年化波动率（Sigma）

    参数:
    - levels: pd.Series，以日期为索引的价格序列
    - day_count: int，年化天数（欧洲市场/日历日通常用 365，股票交易日通常用 252）

    返回:
    - float，年化波动率
    """
    # 1. 严格按时间排序并计算对数收益率
    # 2. 用 replace 把可能产生的无限值(inf)转为 NaN，然后一次性 drop 掉，极具防错性
    log_rets = (
        np.log(levels.sort_index() / levels.shift(1))
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    # 计算样本标准差（ddof=1）并乘以年化系数
    return float(log_rets.std(ddof=1) * np.sqrt(day_count))

## 3.2 (BS)核心参数准备

In [18]:
def get_s0(
    levels: pd.Series,
    start_date: str,
    default_s0: float = 1000.0,
    use_real: bool = False,
) -> Tuple[float, Optional[pd.Timestamp]]:
    """获取期初价格 S0：支持默认标准化基数或按指定日期提取真实历史价。"""
    if use_real:
        d = pd.Timestamp(start_date)
        if d in levels.index:
            return float(levels.loc[d]), d

    return float(default_s0), None


def build_bs_params_simple(
    levels: pd.Series,
    start_date: str,
    r_neutre_annual_cc: float,
    day_count: int = 365,
    s0_default: float = 1000.0,
    use_real_s0: bool = False,
) -> BSParams:
    """
    一键构建 BS 模拟参数：
      - 自动计算历史波动率 (Sigma)
      - 自动获取或指定 S0
      - 组装并返回 BSParams 对象
    """
    s0, s0_date = get_s0(levels, start_date, s0_default, use_real_s0)
    sigma = estimate_sigma_from_history(levels, day_count=day_count)

    return BSParams(
        r_annual_cc=float(r_neutre_annual_cc),
        sigma_annual=float(sigma),
        s0=s0,
        s0_date=s0_date,
    )

In [ ]:
def get_s0(
    levels: pd.Series,
    start_date: str,
    default_s0: float = 1000.0,
    use_real_if_available: bool = False,
) -> tuple[float, Optional[pd.Timestamp]]:
    """Chọn S0: default hoặc lấy đúng level tại start_date nếu có."""
    if not use_real_if_available:
        return float(default_s0), None

    d = pd.Timestamp(start_date)
    if d in levels.index:
        return float(levels.loc[d]), d
    return float(default_s0), None


def build_bs_params_simple(
    levels: pd.Series,
    start_date: str,
    r_neutre_annual_cc: float,
    day_count: int = 365,
    s0_default: float = 1000.0,
    use_real_s0: bool = False,
) -> BSParams:
    """
    Build params đơn giản:
      - sigma: từ dữ liệu quá khứ
      - r_cc: = r_neutre (bạn truyền vào)
      - s0: default hoặc lấy level thật tại start_date nếu có
    """
    s0, s0_date = get_s0(
        levels,
        start_date=start_date,
        default_s0=s0_default,
        use_real_if_available=use_real_s0,
    )

    sigma = estimate_sigma_from_history(levels, day_count=day_count)

    return BSParams(
        r_annual_cc=float(r_neutre_annual_cc),
        sigma_annual=float(sigma),
        s0=float(s0),
        s0_date=s0_date,
    )

In [19]:
r_neutre_cc = 0.0691009521484375

params = build_bs_params_simple(
    levels=levels,
    start_date="2025-12-31",
    r_neutre_annual_cc=r_neutre_cc,
    s0_default=1000,
    use_real_s0=False,
)

print(params)

BSParams(r_annual_cc=0.0691009521484375, sigma_annual=0.23032150063543427, s0=1000.0, s0_date=None)


# 4 - Path GBM




## 4.1 - Theory

#### 1. 核心数学公式回顾

几何布朗运动（GBM）的离散化对数收益率公式为：


$$\ln\left(\frac{S_{t+\Delta t}}{S_t}\right) = \left(r - \frac{1}{2}\sigma^2\right)\Delta t + \sigma \sqrt{\Delta t} Z$$


其中 $Z \sim \mathcal{N}(0, 1)$ 是标准正态分布随机数。

你脑海中浮现的“标准 BS 模型公式”通常指的是它的**微分形式（SDE）**：

$$dS_t = r S_t dt + \sigma S_t dW_t$$

而我在前面代码中写的那行公式：


$$\ln\left(\frac{S_{t+\Delta t}}{S_t}\right) = \left(r - \frac{1}{2}\sigma^2\right)\Delta t + \sigma \sqrt{\Delta t} Z$$

并不是凭空捏造的，它是**从标准的微分形式出发，通过伊藤引理（Itô's Lemma）严格推导出的“离散时间下的精确解”**。

在写蒙特卡洛模拟代码时，我们不能直接用微分的 $dt$（计算机无法处理无穷小），必须离散化。如果用最粗糙的欧拉格式去离散化 $dS_t$，会产生较大的截断误差。因此，金融工程中通常会利用下面这个**对数形式的精确转移方程**来进行模拟。

下面我为你**详细推导证明**这个公式是如何从标准 $dS_t$ 变过来的。

---

### 第一步：写出标准的 Black-Scholes 微分方程

资产价格 $S_t$ 服从几何布朗运动（GBM）：


$$dS_t = r S_t dt + \sigma S_t dW_t$$


其中：

* $r$ 是无风险利率（漂移率）。
* $\sigma$ 是波动率。
* $dW_t \sim \mathcal{N}(0, dt)$ 是维纳过程（布朗运动）的增量。

---

### 第二步：运用伊藤引理（Itô's Lemma）

我们想要研究资产价格的对数变化，因此定义一个新变量：


$$X_t = f(S_t) = \ln(S_t)$$

根据**伊藤引理**，对于函数 $f(S_t)$，它的微分 $df$ 可以展开为泰勒展开式的随机版本：


$$df = \frac{\partial f}{\partial t} dt + \frac{\partial f}{\partial S} dS_t + \frac{1}{2} \frac{\partial^2 f}{\partial S^2} (dS_t)^2$$

我们分别求出各个偏导数：

1. $\frac{\partial f}{\partial t} = 0$ （因为公式里没有显式包含时间 $t$）
2. $\frac{\partial f}{\partial S} = \frac{1}{S_t}$
3. $\frac{\partial^2 f}{\partial S^2} = -\frac{1}{S_t^2}$

把这三个偏导数代入伊藤引理公式中：


$$d(\ln S_t) = \frac{1}{S_t} dS_t - \frac{1}{2 S_t^2} (dS_t)^2$$

---

### 第三步：代入 $dS_t$ 并化简

把标准的 $dS_t = r S_t dt + \sigma S_t dW_t$ 代入上式：

1. 第一项：

$$\frac{1}{S_t} dS_t = \frac{1}{S_t} (r S_t dt + \sigma S_t dW_t) = r dt + \sigma dW_t$$


2. 第二项，我们需要先计算 $(dS_t)^2$。在随机积分的运算法则（Itô multiplication table）中：
* $(dt)^2 = 0$
* $dt \cdot dW_t = 0$
* $(dW_t)^2 = dt$


因此：

$$(dS_t)^2 = (r S_t dt + \sigma S_t dW_t)^2 = r^2 S_t^2 (dt)^2 + 2r\sigma S_t^2 dt dW_t + \sigma^2 S_t^2 (dW_t)^2$$



当取极限时，只有最后一项 $(dW_t)^2 = dt$ 保留下来：

$$(dS_t)^2 = \sigma^2 S_t^2 dt$$


3. 把 $(dS_t)^2$ 代回第二项：

$$-\frac{1}{2 S_t^2} (dS_t)^2 = -\frac{1}{2 S_t^2} (\sigma^2 S_t^2 dt) = -\frac{1}{2} \sigma^2 dt$$



---

### 第四步：合并同类项，得到对数微分方程

把上面各部分加起来：


$$d(\ln S_t) = \left( r dt + \sigma dW_t \right) - \frac{1}{2} \sigma^2 dt$$

重新整理，把 $dt$ 提取公因式：


$$d(\ln S_t) = \left( r - \frac{1}{2} \sigma^2 \right) dt + \sigma dW_t$$

这就是**对数价格的随机微分方程**。你会发现，它变成了一个确定性的漂移项 $\left( r - \frac{1}{2} \sigma^2 \right) dt$ 加上一个常数扩散项 $\sigma dW_t$。

---

### 第五步：从微分到积分（离散化到时间步 $\Delta t$）

现在我们把这个微分方程从当前时间 $t$ 积分到下一个观察时间 $t + \Delta t$：


$$\int_t^{t+\Delta t} d(\ln S_u) = \int_t^{t+\Delta t} \left( r - \frac{1}{2} \sigma^2 \right) du + \int_t^{t+\Delta t} \sigma dW_u$$

左边积分结果为：


$$\ln(S_{t+\Delta t}) - \ln(S_t) = \ln\left(\frac{S_{t+\Delta t}}{S_t}\right)$$

右边由于参数是常数，积分非常简单：

* 第一项积分：$\left( r - \frac{1}{2} \sigma^2 \right) \Delta t$
* 第二项积分：布朗运动的增量 $W_{t+\Delta t} - W_t$ 服从正态分布 $\mathcal{N}(0, \Delta t)$。根据正态分布的性质，它可以写成 $\sqrt{\Delta t} Z$，其中 $Z \sim \mathcal{N}(0, 1)$。

最终，我们就拿到了代码里写的那行公式：


$$\ln\left(\frac{S_{t+\Delta t}}{S_t}\right) = \left( r - \frac{1}{2} \sigma^2 \right) \Delta t + \sigma \sqrt{\Delta t} Z$$

---

### 💡 为什么代码里要用这个公式？

1. **数学上是“精确解”**：它没有做任何近似，是通过伊藤引理严格推导出来的。
2. **避免路径爆炸**：如果在代码里直接对 $S_t$ 用欧拉格式（$S_{t+\Delta t} = S_t + r S_t \Delta t + \dots$），当步长不够小时，指数级增长或随机波动很容易让价格算术溢出或失真。而转到对数空间后，变成了**加法运算**（`np.cumsum`），计算机算起来极其稳定且高效。

## 4.2 - 解析code

#### 2. 代码逐行拆解

* **初始化随机数生成器**：
```python
rng = np.random.default_rng(seed)

```


* 使用 NumPy 推荐的现代 API `default_rng`（比旧版的 `np.random.seed` 更安全、性能更好），并支持固定 `seed` 以确保结果可复现。


* **定义时间步长与日期网格**：
```python
dt = 1.0 / 12.0
dates = pd.date_range(start=pd.Timestamp(start_date), periods=n_months + 1, freq="M")

```


* `dt = 1/12` 代表按月模拟（一年 12 个月）。
* `pd.date_range` 生成从起点开始、包含 `n_months + 1` 个月的月末日期序列（包含 $t=0$ 的初始日）。


* **生成标准正态随机数矩阵**：
```python
Z = rng.standard_normal(size=(n_months, n_sims))

```


* 生成一个形状为 `(n_months, n_sims)` 的二维矩阵。**行代表时间步（第 1 个月到第 $N$ 个月），列代表不同的模拟路径（情景）**。


* **计算漂移项与扩散项**：
```python
drift = (r_annual_cc - 0.5 * sigma_annual**2) * dt
diffusion = sigma_annual * np.sqrt(dt) * Z

```


* `drift` 是确定性的漂移率（广播机制会让它自动适配整个矩阵）。
* `diffusion` 是随机震荡项，每一期乘上 $\sqrt{dt}$ 和随机数 $Z$。


* **利用对数的可加性计算累积路径 (`np.cumsum`)**：
这两行代码是整个蒙特卡洛模拟中最巧妙、最精髓的部分！

直接回答你的问题：**`paths` 对应的是 $S_t$（资产在各个未来时点的绝对价格水平）**，**绝对不是** $dS_t$。$dS_t$ 只是价格的微小变化量，而 `paths` 算出来的是我们最终要用的**真实资产价格矩阵**。

为了让你彻底看懂，我们把这两行代码拆开，结合刚才推导的对数公式，一步步看它在数学上到底做了什么：

---

### 第一行代码拆解：`log_paths`

```python
log_paths = np.vstack([np.zeros((1, n_sims)), np.cumsum(drift + diffusion, axis=0)])

```

1. **`drift + diffusion` 是什么？**
这对应着我们刚才推导的单步对数收益率增量（即经过一个时间步长 $\Delta t$ 后的变化）：

$$\Delta \ln(S) = \left( r - \frac{1}{2}\sigma^2 \right)\Delta t + \sigma \sqrt{\Delta t} Z$$



它代表的是“单个月份”内价格对数的变化量。
2. **`np.cumsum(..., axis=0)` 的数学含义（核心！）：**
* **`cumsum`（Cumulative Sum）** 是**累积求和**的意思。
* 为什么要累积？因为对数（$\ln$）有一个极强的数学性质——**可加性**。从今天（$t=0$）到第 3 个月的总对数收益率，等于第 1 个月、第 2 个月、第 3 个月单步对数收益率的**相加**：

$$\ln\left(\frac{S_{t_3}}{S_0}\right) = \ln\left(\frac{S_{t_1}}{S_0}\right) + \ln\left(\frac{S_{t_2}}{S_{t_1}}\right) + \ln\left(\frac{S_{t_3}}{S_{t_2}}\right)$$


* 因此，`np.cumsum` 沿时间轴（`axis=0`）一加，算出来的 `log_paths` 实际上存的是：**从期初 $S_0$ 到未来每个月 $S_t$ 的“总对数收益率”**，即：

$$\text{log\_path}_t = \ln\left(\frac{S_t}{S_0}\right)$$


* 最前面的 `np.zeros((1, n_sims))` 是为了补上 $t=0$ 时的初始状态（因为刚开始时 $\ln(S_0 / S_0) = 0$）。



---

### 第二行代码拆解：`paths`

```python
paths = s0 * np.exp(log_paths)

```

既然我们在上一步通过累加算出了对数比值 $\ln\left(\frac{S_t}{S_0}\right)$，那怎么把它还原成真正的价格 $S_t$ 呢？

1. **取指数（`np.exp`）**：
数学上，指数函数 $\exp$ 是对数函数 $\ln$ 的逆运算。如果对两边同时取指数：

$$\exp\left[ \ln\left(\frac{S_t}{S_0}\right) \right] = \frac{S_t}{S_0}$$



这样就把讨厌的对数消掉了，得到了价格相对于初始价的**倍数**（涨跌幅比例）。
2. **乘以期初价格 `s0` ($S_0$)**：

$$\text{paths} = S_0 \times \frac{S_t}{S_0} = S_t$$



这就完美还原出了**未来每个时间点上，资产的绝对价格 $S_t$**！

---

### 💡 总结成一幅图

这段代码在计算机里构建的矩阵逻辑是这样的：

* **第 0 行 ($t=0$)**：全都是 $S_0$（比如 1000, 1000, 1000...）
* **第 1 个月 ($t=1$)**：通过 $\ln$ 的累加和指数还原，算出各条路径在第 1 个月的真实价格（比如 1020, 985, 1011...）
* **第 2 个月 ($t=2$)**：算出第 2 个月的真实价格...
* ……一直算到最后一个月。

所以，`paths` 就是最终生成的“多条资产价格模拟路径表”，每一行代表一个情景（Scenario），每一列代表一个未来的月份，表格里的每一个数字就是对应的 $S_t$！


* **转置并组装成 DataFrame**：
```python
df = pd.DataFrame(paths.T, index=np.arange(1, n_sims + 1), columns=dates)
df.index.name = "scenario"
return df

```


* `paths.T` 把矩阵转置，变成 **行是情景（`n_sims`），列是日期（`n_months + 1`）**，符合金融回测中“每一行代表一条完整模拟路径”的习惯。





In [20]:
def simulate_gbm_monthly(
    s0: float,
    r_annual_cc: float,
    sigma_annual: float,
    start_date: str,
    n_months: int,
    n_sims: int,
    seed: Optional[int] = 42,
) -> pd.DataFrame:
    """
    模块 4：基于几何布朗运动 (GBM) 的月度资产路径模拟
    采用 NumPy 内存预分配与对数累加法，性能极致优化。
    返回:
        pd.DataFrame (行 = 情景 scenario，列 = 月末日期 dates)
    """
    if s0 <= 0 or n_months <= 0 or n_sims <= 0:
        raise ValueError("参数 s0、n_months、n_sims 必须大于 0。")

    rng = np.random.default_rng(seed)
    dt = 1.0 / 12.0

    # 内存预分配：(n_months + 1, n_sims)
    log_paths = np.zeros((n_months + 1, n_sims))

    drift = (r_annual_cc - 0.5 * sigma_annual ** 2) * dt
    diffusion_coeff = sigma_annual * np.sqrt(dt)

    # 生成随机数并利用对数可加性进行累加
    z = rng.standard_normal(size=(n_months, n_sims))
    log_paths[1:] = np.cumsum(drift + diffusion_coeff * z, axis=0)

    # 指数还原为绝对价格
    paths = s0 * np.exp(log_paths)
    dates = pd.date_range(start=pd.Timestamp(start_date), periods=n_months + 1, freq="ME")

    return pd.DataFrame(
        paths.T,
        index=pd.Index(range(1, n_sims + 1), name="scenario"),
        columns=dates
    )

In [23]:
    paths = simulate_gbm_monthly(
        s0=params.s0,
        r_annual_cc=params.r_annual_cc,
        sigma_annual=params.sigma_annual,
        start_date="2025-12-31",
        n_months=145,
        n_sims=10_000,
        seed=42,
    )
    print(paths)

          2025-12-31   2026-01-31   2026-02-28   2026-03-31   2026-04-30  \
scenario                                                                   
1             1000.0  1024.093813  1039.850070   934.609975   876.932036   
2             1000.0   936.506817   997.733179  1062.403013  1062.824567   
3             1000.0  1054.898070   959.928731   959.630856   880.578854   
4             1000.0  1068.316913  1051.968931  1014.427941   970.746690   
5             1000.0   881.462760   934.731810   922.540572   910.909562   
...              ...          ...          ...          ...          ...   
9996          1000.0  1117.691998  1261.313979  1284.614594  1204.849517   
9997          1000.0   999.347586   950.182912   965.698522   901.018011   
9998          1000.0  1009.258707  1056.967334  1201.003574  1334.034050   
9999          1000.0  1081.479675  1124.328844  1157.295685  1093.190521   
10000         1000.0   991.427693  1075.796217  1025.002359  1002.610576   

           

# 5 - decrement

In [ ]:

from typing import Literal
DecrementMode = Literal["flat", "cum"]

def decrement_paths(
    df: pd.DataFrame,
    decrement_value: float = 50.0,   # 50 points per year
    periods_per_year: int = 12,      # monthly
    mode: DecrementMode = "cum",     # với logic bạn nêu: dùng "cum"
) -> pd.DataFrame:
    """
    Linear decrement per period with cumulative option.

    Your target behavior (mode="cum", monthly):
      col0 (t0): 0
      col1:  -50/12
      col2:  -2*50/12
      ...
      colk:  -k*50/12

    mode:
      - "flat": subtract 50/12 mỗi kỳ (không tích lũy)
      - "cum" : subtract k*(50/12) (tích lũy theo số kỳ)
    """
    df_dec = df.copy()

    # ensure datetime columns + sort
    cols = pd.to_datetime(df_dec.columns)
    order = np.argsort(cols.values)
    df_dec = df_dec.iloc[:, order]

    n_cols = df_dec.shape[1]
    if n_cols == 0:
        return df_dec

    per_period = float(decrement_value) / float(periods_per_year)  # 50/12

    # k = 0,1,2,...,n_cols-1 where col0 is t0
    k = np.arange(n_cols, dtype=float)

    if mode == "flat":
        decrements = per_period * (k > 0)          # col0=0, các cột sau trừ 50/12
    elif mode == "cum":
        decrements = per_period * k                # col0=0, col1=1*50/12, col2=2*50/12,...
    else:
        raise ValueError("mode must be 'flat' or 'cum'")

    df_dec.iloc[:, :] = df_dec.values - decrements
    return df_dec

In [ ]:
path_decrement = decrement_paths(paths, decrement_value=50, mode = 'cum')

In [ ]:
path_decrement.head()

,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30,2026-07-31,2026-08-31,2026-09-30,...,2037-04-30,2037-05-31,2037-06-30,2037-07-31,2037-08-31,2037-09-30,2037-10-31,2037-11-30,2037-12-31,2038-01-31
scenario,,,,,,,,,,,,,,,,,,,,,
1,1000.0,1021.696203,1034.126660,912.614746,844.474973,787.396035,833.095952,882.747050,766.884344,805.127221,...,1075.777219,936.918683,666.873096,655.707250,540.795634,580.067027,638.647134,644.587937,681.642614,731.405525
2,1000.0,924.720891,987.337480,1054.131773,1049.465620,953.655082,828.015882,806.995786,847.068254,889.294136,...,259.542262,110.958383,141.779543,164.361785,293.664158,274.851697,303.140505,247.839118,305.945257,284.455953
3,1000.0,1056.025361,945.524691,940.162903,848.453382,868.491560,851.377087,954.848710,977.602081,1009.232103,...,1125.126238,1099.468857,1239.884122,1105.557211,1100.726005,994.006986,1055.008679,1093.653297,1112.585885,1257.756984
4,1000.0,1071.014629,1047.629230,1000.770491,947.383763,957.156468,880.050892,813.432963,773.281387,879.198614,...,765.904285,612.716293,486.053586,427.088524,629.977585,637.604131,676.520777,769.216775,627.213774,529.163620
5,1000.0,864.283003,917.756802,899.355275,881.611988,743.677259,922.416480,810.198310,739.707322,716.042935,...,5218.182124,4780.917968,4744.292689,5111.755243,5055.068847,5306.201154,5234.918772,4487.499075,4491.506402,4469.133238
